<a href="https://colab.research.google.com/github/ratneshpal700-ops/celebal-project/blob/main/Assignment_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Libraries



In [30]:
!pip install -q langchain langchain-community langchain-text-splitters pypdf sentence-transformers faiss-cpu transformers accelerate

## Import Libraries

In [31]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import torch
from transformers import pipeline

print("All imports successful!")


All imports successful!


In [32]:
from google.colab import files

uploaded = files.upload()

Saving About Prasuta Ventures Ecosystem.pdf to About Prasuta Ventures Ecosystem.pdf


In [33]:
import os

# Grab the filename dynamically instead of hardcoding it,
# so this works regardless of what file name you upload.
pdf_filename = list(uploaded.keys())[0]
print("Uploaded file:", pdf_filename)


Uploaded file: About Prasuta Ventures Ecosystem.pdf


In [34]:
loader = PyPDFLoader(pdf_filename)

documents = loader.load()

print(documents[0].page_content)


Understanding the Prasuta Ventures 
Ecosystem 
Executive Summary 
Prasuta Ventures is a diversified ecosystem that operates across education, finance, talent 
development, and social impact. Rather than functioning as a single business entity, it has 
created multiple verticals that work together to provide learning opportunities, professional 
development, investment expertise, and community support. This report is based on 
research conducted through the official website, LinkedIn profiles, Instagram pages, and 
other publicly available information related to the organization. 
The four key verticals of the ecosystem are Prasuta FinSchool, Prasuta Capital, Prasuta Core, 
and Prasuta Foundation. Each vertical serves a different purpose while contributing to the 
overall vision of empowering individuals and creating long-term value for society. 
1. Prasuta FinSchool 
Overview  
Prasuta FinSchool is the education and skill development division of Prasuta Ventures. It 
focuses on prepari

## Split the Text into Chunks

In [35]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100  # matches the value documented in the report
)

chunks = text_splitter.split_documents(
    documents
)

print(
    "Number of Chunks:",
    len(chunks)
)


Number of Chunks: 25


## Convert Chunks into Text

In [36]:
texts = [
    chunk.page_content
    for chunk in chunks
]

print(texts[0])

Understanding the Prasuta Ventures 
Ecosystem 
Executive Summary 
Prasuta Ventures is a diversified ecosystem that operates across education, finance, talent 
development, and social impact. Rather than functioning as a single business entity, it has 
created multiple verticals that work together to provide learning opportunities, professional 
development, investment expertise, and community support. This report is based on


## Create Embeddings

In [37]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = model.encode(
    texts,
    convert_to_numpy=True
)

print(
    embeddings.shape
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(25, 384)


## Create FAISS Vector Database

In [38]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    np.array(embeddings).astype("float32")
)

print(
    "Embeddings Stored Successfully!"
)


Embeddings Stored Successfully!


## Ask a Question

In [46]:
query = """What are the eco system of prasuta""".strip()

print(query)


What are the eco system of prasuta


## Convert Question to Embedding

In [47]:
query_embedding = model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")


## Retrieve Top 3 Chunks

In [48]:
k = 3

distances, indices = index.search(
    query_embedding,
    k
)

print("Indices:", indices)
print("Distances:", distances)


Indices: [[20 12 11]]
Distances: [[0.85758066 0.9022911  0.91305566]]


## Display Retrieved Chunks

In [49]:
retrieved_chunks = []

for idx in indices[0]:
    retrieved_chunks.append(
        texts[idx]
    )

for chunk in retrieved_chunks:
    print(chunk)
    print("-"*50)

Prasuta Foundation represents the social responsibility aspect of the ecosystem. It extends 
the organization's impact beyond business objectives and contributes to community welfare 
and inclusive development. 
Ecosystem Analysis: How the Verticals Work Together 
The four verticals of Prasuta Ventures are interconnected and support one another. 
• Prasuta FinSchool develops knowledge, technical skills, and employability. 
• Prasuta Core provides internships, networking, and industry exposure.
--------------------------------------------------
Overview  
Prasuta Core focuses on talent development, networking, internship opportunities, and 
ecosystem building. It connects students, professionals, startups, and industry experts.
--------------------------------------------------
strong emphasis on research-based decision-making. 
Analysis and Understanding  
Prasuta Capital serves as the knowledge and research center of the ecosystem. It contributes 
financial expertise and analytical ca

## Load the Language Model


In [50]:
from transformers import pipeline

# Although FLAN-T5 is a sequence-to-sequence (encoder-decoder) model,
# the 'text2text-generation' task is not recognized by the current version
# of the `transformers` library as per the KeyError. Using 'text-generation'
# instead, as it is an available generic text generation task. The pipeline
# should correctly handle FLAN-T5's architecture.
device = 0 if torch.cuda.is_available() else -1  # use the T4 GPU if available

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    device=device
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM',

## Prompt

In [52]:
context = "\n\n".join(retrieved_chunks)

prompt = f"""Answer the question using ONLY the context below.
If the answer is not contained in the context, say "I don't know".

Context:
{context}

Question:
{query}

Answer:"""


## Final Answer

In [53]:
answer = generator(
    prompt,
    max_new_tokens=150
)

print(answer[0]["generated_text"].strip())


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer the question using ONLY the context below.
If the answer is not contained in the context, say "I don't know".

Context:
Prasuta Foundation represents the social responsibility aspect of the ecosystem. It extends 
the organization's impact beyond business objectives and contributes to community welfare 
and inclusive development. 
Ecosystem Analysis: How the Verticals Work Together 
The four verticals of Prasuta Ventures are interconnected and support one another. 
• Prasuta FinSchool develops knowledge, technical skills, and employability. 
• Prasuta Core provides internships, networking, and industry exposure.

Overview  
Prasuta Core focuses on talent development, networking, internship opportunities, and 
ecosystem building. It connects students, professionals, startups, and industry experts.

strong emphasis on research-based decision-making. 
Analysis and Understanding  
Prasuta Capital serves as the knowledge and research center of the ecosystem. It contributes 
financial 

## Conclusion

This project successfully developed and implemented a Retrieval-Augmented Generation (RAG) system that combines document retrieval and natural language generation to provide accurate and context-aware answers from custom PDF documents. The pipeline was built using LangChain for workflow management, Sentence Transformers for generating semantic embeddings, FAISS for efficient similarity-based document retrieval, and FLAN-T5 as the language generation model.

The system effectively extracts relevant information from the uploaded documents, retrieves the most meaningful content based on user queries, and uses the language model to generate precise responses. This approach reduces the chances of generating irrelevant or incorrect answers by grounding the responses in the provided document knowledge.

Overall, the project demonstrates the effectiveness of RAG architecture in building intelligent question-answering systems. It can be further enhanced by integrating larger language models, improving document preprocessing techniques, supporting multiple document formats, and optimizing retrieval methods for better accuracy and scalability. This implementation provides a strong foundation for developing real-world AI-powered knowledge retrieval applications.
